In [ ]:
# ============================================================
# CELL 1 — INSTALL + IMPORT
# ============================================================
!pip install -q transformers accelerate bitsandbytes sentencepiece datasets==2.21.0 rouge_score fuzzywuzzy python-Levenshtein rouge

import os, gc, json, math, types, subprocess, sys
from collections import defaultdict, Counter
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.models.llama.modeling_llama import LlamaAttention, apply_rotary_pos_emb

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
GPU: Tesla T4
PyTorch: 2.11.0+cu130


In [ ]:
# ============================================================
# CELL 2 — CONFIG + LONGBENCH + MODEL
# ============================================================
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
TASKS = ["narrativeqa", "qasper", "gov_report"]
SAMPLES_PER_TASK = 15
MAX_TOTAL_SAMPLES = 45
SEQ_LEN = 2048
MAX_NEW_TOKENS_DEFAULT = 64
BLOCK_ROWS = 32
BLOCK_COLS = 32
SPARSITY_LEVELS = [50.0, 70.0, 80.0, 90.0]
PROXY_NAME = "Frobenius(Q_block) * Frobenius(K_block)"

if not os.path.exists("/content/LongBench"):
    !git clone -q https://github.com/THUDM/LongBench.git /content/LongBench
else:
    print("LongBench already exists")

from huggingface_hub import login
login()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.eval()
device = next(model.parameters()).device

samples = []
for task in TASKS:
    print("Loading:", task)
    ds = load_dataset("THUDM/LongBench", task, split="test")
    n = min(SAMPLES_PER_TASK, len(ds))
    for item in ds.select(range(n)):
        samples.append({
            "task": task,
            "context": item["context"],
            "input": item["input"],
            "answers": item["answers"],
            "all_classes": item.get("all_classes", []),
            "length": item.get("length", None),
        })
samples = samples[:MAX_TOTAL_SAMPLES]

CONFIG_DIR = "/content/LongBench/LongBench/config"
with open(os.path.join(CONFIG_DIR, "dataset2prompt.json"), encoding="utf-8") as f:
    dataset2prompt = json.load(f)
maxlen_path = os.path.join(CONFIG_DIR, "dataset2maxlen.json")
if os.path.exists(maxlen_path):
    with open(maxlen_path, encoding="utf-8") as f:
        dataset2maxlen = json.load(f)
else:
    dataset2maxlen = {}

os.makedirs("/content/pred", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

print("Total samples:", len(samples))
print("By task:", Counter(s["task"] for s in samples))
print("Model:", MODEL_NAME)
print("Device:", device)
print("Block:", f"{BLOCK_ROWS}x{BLOCK_COLS}")
print("Proxy:", PROXY_NAME)


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

Loading: qasper


Generating test split: 0 examples [00:00, ? examples/s]

Loading: gov_report


Generating test split: 0 examples [00:00, ? examples/s]

Total samples: 45
By task: Counter({'narrativeqa': 15, 'qasper': 15, 'gov_report': 15})
Model: meta-llama/Llama-2-7b-chat-hf
Device: cuda:0
Block: 32x32
Proxy: Frobenius(Q_block) * Frobenius(K_block)


In [ ]:
# ============================================================
# CELL 3 — CLEAN NORM-PROXY ATTENTION PATCH
# ============================================================
#
# IMPORTANT:
# This cell deliberately restores the ORIGINAL class-level
# LlamaAttention.forward before installing the new patch.
#
# This prevents old monkey-patched forward functions from
# remaining active in the notebook.
#
# Method:
#
#   Q/K
#    ↓
#   RoPE
#    ↓
#   ||Q_block||F × ||K_block||F
#    ↓
#   Top-K selected blocks
#    ↓
#   EXACT QK ONLY FOR SELECTED BLOCKS
#    ↓
#   causal mask
#    ↓
#   softmax
#    ↓
#   selected V
#    ↓
#   attention × V
#
# NO:
#   - L1(QK)
#   - full QK matrix
#   - L1 ranking
#
# ============================================================

import math
import types
import torch


# ============================================================
# CONFIG
# ============================================================

BLOCK_ROWS = 32
BLOCK_COLS = 32

CURRENT_PROXY_SPARSITY = 50.0


# ============================================================
# STATISTICS
# ============================================================

proxy_stats = {}


def reset_proxy_stats():

    global proxy_stats

    proxy_stats = {
        "candidate_blocks": 0,
        "kept_blocks": 0,
        "threshold_sum": 0.0,
        "threshold_count": 0,
        "attention_calls": 0,
    }


# ============================================================
# NORM-PROXY ESTIMATOR
# ============================================================
#
# Proxy:
#
#       P(i,j) = ||Q_i||F * ||K_j||F
#
# No QK^T is computed here.
#
# ============================================================

def get_norm_proxy_mask(
    q,
    k,
    target_sparsity,
    block_rows=BLOCK_ROWS,
    block_cols=BLOCK_COLS,
):

    q_len, head_dim = q.shape
    k_len, _ = k.shape

    num_q_blocks = (
        q_len // block_rows
    )

    num_k_blocks = (
        k_len // block_cols
    )

    keep_mask = torch.zeros(
        num_q_blocks,
        num_k_blocks,
        dtype=torch.bool,
        device=q.device,
    )

    if (
        num_q_blocks == 0
        or num_k_blocks == 0
    ):

        return (
            keep_mask,
            0,
            0,
            float("nan"),
        )

    # --------------------------------------------------------
    # Q block norms
    # --------------------------------------------------------

    q_blocks = q[
        :num_q_blocks * block_rows,
        :
    ].reshape(
        num_q_blocks,
        block_rows,
        head_dim,
    )

    q_norms = torch.linalg.vector_norm(
        q_blocks,
        ord=2,
        dim=(1, 2),
    )

    # --------------------------------------------------------
    # K block norms
    # --------------------------------------------------------

    k_blocks = k[
        :num_k_blocks * block_cols,
        :
    ].reshape(
        num_k_blocks,
        block_cols,
        head_dim,
    )

    k_norms = torch.linalg.vector_norm(
        k_blocks,
        ord=2,
        dim=(1, 2),
    )

    # --------------------------------------------------------
    # Proxy importance
    #
    # [Q_blocks, 1]
    #       *
    # [1, K_blocks]
    #
    # -> [Q_blocks, K_blocks]
    # --------------------------------------------------------

    proxy_matrix = (
        q_norms[:, None]
        * k_norms[None, :]
    )

    # --------------------------------------------------------
    # Causal candidate blocks
    #
    # A block is a candidate if at least one element in the
    # block is causally valid.
    # --------------------------------------------------------

    q_end = (
        torch.arange(
            num_q_blocks,
            device=q.device,
        )
        + 1
    ) * block_rows

    k_start = (
        torch.arange(
            num_k_blocks,
            device=q.device,
        )
        * block_cols
    )

    candidate_mask = (
        k_start[None, :]
        < q_end[:, None]
    )

    candidate_scores = proxy_matrix[
        candidate_mask
    ]

    candidate_count = int(
        candidate_scores.numel()
    )

    if candidate_count == 0:

        return (
            keep_mask,
            0,
            0,
            float("nan"),
        )

    # --------------------------------------------------------
    # Keep ratio
    #
    # sparsity 90% -> keep 10%
    # sparsity 80% -> keep 20%
    # sparsity 70% -> keep 30%
    # sparsity 50% -> keep 50%
    # --------------------------------------------------------

    keep_ratio = max(
        0.0,
        min(
            1.0,
            1.0
            - float(target_sparsity)
            / 100.0,
        ),
    )

    keep_count = int(
        round(
            candidate_count
            * keep_ratio
        )
    )

    # At least one block must remain.
    keep_count = max(
        1,
        min(
            candidate_count,
            keep_count,
        ),
    )

    # --------------------------------------------------------
    # TOP-K
    # --------------------------------------------------------

    top_values, top_indices = torch.topk(
        candidate_scores,
        k=keep_count,
        largest=True,
        sorted=False,
    )

    threshold = float(
        top_values.min().item()
    )

    # --------------------------------------------------------
    # Convert flattened candidate indices
    # back to [block_row, block_col].
    # --------------------------------------------------------

    candidate_positions = (
        candidate_mask.nonzero(
            as_tuple=False
        )
    )

    selected_positions = (
        candidate_positions[
            top_indices
        ]
    )

    keep_mask[
        selected_positions[:, 0],
        selected_positions[:, 1],
    ] = True

    return (
        keep_mask,
        candidate_count,
        keep_count,
        threshold,
    )


# ============================================================
# RESTORE TRUE ORIGINAL FORWARD
# ============================================================
#
# IMPORTANT:
#
# DO NOT restore from "_proxy_original_forward".
#
# That variable may already contain an OLD PATCH because this
# notebook has been patched multiple times.
#
# Instead, restore the class-level method directly.
#
# ============================================================

def restore_original_attention(model):

    restored = 0

    for layer in model.model.layers:

        attn = layer.self_attn

        cls_forward = type(attn).forward

        attn.forward = types.MethodType(
            cls_forward,
            attn,
        )

        restored += 1

    print(
        f"Restored original LlamaAttention.forward "
        f"for {restored} layers."
    )


# ============================================================
# CACHE SEQUENCE LENGTH
# ============================================================

def get_cache_seq_length(
    past,
    layer_idx,
):

    if past is None:

        return 0

    # --------------------------------------------------------
    # Modern Cache API
    # --------------------------------------------------------

    if hasattr(
        past,
        "get_seq_length",
    ):

        try:

            return int(
                past.get_seq_length(
                    layer_idx
                )
            )

        except TypeError:

            return int(
                past.get_seq_length()
            )

    # --------------------------------------------------------
    # Legacy tuple cache
    # --------------------------------------------------------

    if isinstance(
        past,
        (tuple, list),
    ):

        if len(past) == 0:

            return 0

        try:

            return int(
                past[0].shape[-2]
            )

        except Exception:

            return 0

    return 0


# ============================================================
# INSTALL NORM-PROXY ATTENTION
# ============================================================

def install_norm_proxy_attention(
    model,
    target_sparsity,
):

    global CURRENT_PROXY_SPARSITY

    CURRENT_PROXY_SPARSITY = float(
        target_sparsity
    )

    # --------------------------------------------------------
    # CRITICAL:
    #
    # Restore REAL class forward first.
    #
    # This eliminates every stale instance-level patch.
    # --------------------------------------------------------

    restore_original_attention(
        model
    )

    reset_proxy_stats()

    # --------------------------------------------------------
    # Install new forward
    # --------------------------------------------------------

    for layer_idx, layer in enumerate(
        model.model.layers
    ):

        attn = layer.self_attn

        # ====================================================
        # CREATE FORWARD FOR THIS LAYER
        # ====================================================

        def make_forward(
            current_layer_idx
        ):

            def proxy_forward(
                self,
                hidden_states,
                position_embeddings=None,
                attention_mask=None,
                past_key_values=None,
                **kwargs,
            ):

                # =================================================
                # SHAPE
                # =================================================

                bsz, q_len, _ = (
                    hidden_states.shape
                )

                # =================================================
                # CACHE LENGTH
                # =================================================

                past_len = (
                    get_cache_seq_length(
                        past_key_values,
                        current_layer_idx,
                    )
                )

                # =================================================
                # DECODE / UNSUPPORTED CASE
                #
                # If this is a real decode step, use the original
                # Hugging Face attention implementation.
                #
                # =================================================

                if (
                    q_len < BLOCK_ROWS
                    or past_len > 0
                ):

                    original_forward = (
                        type(self).forward
                    )

                    return original_forward(
                        self,
                        hidden_states,
                        position_embeddings=position_embeddings,
                        attention_mask=attention_mask,
                        past_key_values=past_key_values,
                        **kwargs,
                    )

                # =================================================
                # PREFILL PROXY PATH
                # =================================================

                if bsz != 1:

                    # Current experiment is batch-size 1.
                    original_forward = (
                        type(self).forward
                    )

                    return original_forward(
                        self,
                        hidden_states,
                        position_embeddings=position_embeddings,
                        attention_mask=attention_mask,
                        past_key_values=past_key_values,
                        **kwargs,
                    )

                # =================================================
                # Q / K / V PROJECTION
                # =================================================

                q = self.q_proj(
                    hidden_states
                )

                k = self.k_proj(
                    hidden_states
                )

                v = self.v_proj(
                    hidden_states
                )

                # =================================================
                # HEAD CONFIG
                # =================================================

                num_heads = int(
                    getattr(
                        self,
                        "num_heads",
                        self.config.num_attention_heads,
                    )
                )

                num_kv_heads = int(
                    getattr(
                        self,
                        "num_key_value_heads",
                        self.config.num_key_value_heads,
                    )
                )

                head_dim = int(
                    getattr(
                        self,
                        "head_dim",
                        self.config.hidden_size
                        // num_heads,
                    )
                )

                # =================================================
                # [B,T,H,D] -> [B,H,T,D]
                # =================================================

                q = (
                    q.view(
                        bsz,
                        q_len,
                        num_heads,
                        head_dim,
                    )
                    .transpose(1, 2)
                    .contiguous()
                )

                k = (
                    k.view(
                        bsz,
                        q_len,
                        num_kv_heads,
                        head_dim,
                    )
                    .transpose(1, 2)
                    .contiguous()
                )

                v = (
                    v.view(
                        bsz,
                        q_len,
                        num_kv_heads,
                        head_dim,
                    )
                    .transpose(1, 2)
                    .contiguous()
                )

                # =================================================
                # RoPE
                #
                # position_embeddings is already generated by
                # LlamaModel and should be used directly.
                # =================================================

                if (
                    position_embeddings
                    is None
                ):

                    raise RuntimeError(
                        "position_embeddings is None "
                        "inside norm-proxy attention."
                    )

                cos, sin = (
                    position_embeddings[:2]
                )

                q, k = apply_rotary_pos_emb(
                    q,
                    k,
                    cos,
                    sin,
                )

                # =================================================
                # EMPTY CACHE PREFILL
                #
                # If use_cache created an empty DynamicCache,
                # update it here just like original LlamaAttention.
                #
                # This keeps generation compatible.
                # =================================================

                if (
                    past_key_values
                    is not None
                ):

                    cache_kwargs = {}

                    # Newer versions may accept these.
                    cache_kwargs[
                        "cos"
                    ] = cos

                    cache_kwargs[
                        "sin"
                    ] = sin

                    try:

                        k, v = (
                            past_key_values.update(
                                k,
                                v,
                                current_layer_idx,
                                cache_kwargs,
                            )
                        )

                    except TypeError:

                        k, v = (
                            past_key_values.update(
                                k,
                                v,
                                current_layer_idx,
                            )
                        )

                # =================================================
                # GQA
                # =================================================

                if (
                    num_heads
                    % num_kv_heads
                    != 0
                ):

                    raise RuntimeError(
                        "Q heads are not divisible by "
                        "KV heads: "
                        f"{num_heads} vs {num_kv_heads}"
                    )

                repeat_factor = (
                    num_heads
                    // num_kv_heads
                )

                if repeat_factor > 1:

                    k = k.repeat_interleave(
                        repeat_factor,
                        dim=1,
                    )

                    v = v.repeat_interleave(
                        repeat_factor,
                        dim=1,
                    )

                key_len = int(
                    k.shape[-2]
                )

                # =================================================
                # SAFETY
                #
                # Proxy is only valid when Q and K lengths match.
                # =================================================

                if key_len != q_len:

                    original_forward = (
                        type(self).forward
                    )

                    return original_forward(
                        self,
                        hidden_states,
                        position_embeddings=position_embeddings,
                        attention_mask=attention_mask,
                        past_key_values=past_key_values,
                        **kwargs,
                    )

                # =================================================
                # OUTPUT
                # =================================================

                output = torch.zeros(
                    (
                        bsz,
                        num_heads,
                        q_len,
                        head_dim,
                    ),
                    device=q.device,
                    dtype=q.dtype,
                )

                num_q_blocks = (
                    q_len
                    // BLOCK_ROWS
                )

                num_k_blocks = (
                    key_len
                    // BLOCK_COLS
                )

                scale = (
                    head_dim ** -0.5
                )

                # =================================================
                # HEAD LOOP
                # =================================================

                for h in range(
                    num_heads
                ):

                    q_head = q[
                        0,
                        h,
                    ]

                    k_head = k[
                        0,
                        h,
                    ]

                    v_head = v[
                        0,
                        h,
                    ]

                    # =================================================
                    # NORM PROXY
                    # =================================================

                    (
                        keep_mask,
                        candidate_count,
                        keep_count,
                        threshold,
                    ) = get_norm_proxy_mask(
                        q_head,
                        k_head,
                        CURRENT_PROXY_SPARSITY,
                        BLOCK_ROWS,
                        BLOCK_COLS,
                    )

                    proxy_stats[
                        "candidate_blocks"
                    ] += candidate_count

                    proxy_stats[
                        "kept_blocks"
                    ] += keep_count

                    proxy_stats[
                        "attention_calls"
                    ] += 1

                    if math.isfinite(
                        threshold
                    ):

                        proxy_stats[
                            "threshold_sum"
                        ] += threshold

                        proxy_stats[
                            "threshold_count"
                        ] += 1

                    # =================================================
                    # Q-BLOCK LOOP
                    # =================================================

                    for br in range(
                        num_q_blocks
                    ):

                        qs = (
                            br
                            * BLOCK_ROWS
                        )

                        qe = (
                            qs
                            + BLOCK_ROWS
                        )

                        # ------------------------------------------------
                        # IMPORTANT:
                        #
                        # selected_bc is ALWAYS a 1-D tensor.
                        # ------------------------------------------------

                        selected_bc = torch.where(
                            keep_mask[br]
                        )[0].reshape(
                            -1
                        )

                        # ------------------------------------------------
                        # Safety fallback.
                        # ------------------------------------------------

                        if selected_bc.numel() == 0:

                            fallback_bc = min(
                                br,
                                num_k_blocks - 1,
                            )

                            selected_bc = torch.tensor(
                                [fallback_bc],
                                device=q.device,
                                dtype=torch.long,
                            )

                        num_selected = int(
                            selected_bc.shape[0]
                        )

                        # =================================================
                        # Q BLOCK
                        # =================================================

                        q_block = q_head[
                            qs:qe,
                            :
                        ]

                        # =================================================
                        # K INDICES
                        #
                        # [N]
                        #    ↓
                        # [N,32]
                        # =================================================

                        k_starts = (
                            selected_bc
                            * BLOCK_COLS
                        )

                        k_offsets = torch.arange(
                            BLOCK_COLS,
                            device=q.device,
                            dtype=torch.long,
                        )

                        k_indices = (
                            k_starts[:, None]
                            + k_offsets[None, :]
                        )

                        # =================================================
                        # GATHER K
                        #
                        # [N*32,D]
                        #    ↓
                        # [N,32,D]
                        # =================================================

                        k_selected = torch.index_select(
                            k_head,
                            0,
                            k_indices.reshape(-1),
                        )

                        k_selected = (
                            k_selected.reshape(
                                num_selected,
                                BLOCK_COLS,
                                head_dim,
                            )
                        )

                        # =================================================
                        # EXPAND Q
                        #
                        # [32,D]
                        #    ↓
                        # [N,32,D]
                        # =================================================

                        q_expand = (
                            q_block
                            .unsqueeze(0)
                            .expand(
                                num_selected,
                                BLOCK_ROWS,
                                head_dim,
                            )
                        )

                        # =================================================
                        # EXACT SELECTED QK
                        #
                        # q_expand:
                        #     [N,32,D]
                        #
                        # k_selected:
                        #     [N,32,D]
                        #
                        # result:
                        #     [N,32,32]
                        #
                        # THIS IS THE ONLY EXACT QK COMPUTATION.
                        # =================================================

                        block_scores = torch.bmm(
                            q_expand,
                            k_selected.transpose(
                                1,
                                2,
                            ),
                        ) * scale

                        # =================================================
                        # HARD ASSERT
                        # =================================================

                        expected_shape = (
                            num_selected,
                            BLOCK_ROWS,
                            BLOCK_COLS,
                        )

                        if tuple(
                            block_scores.shape
                        ) != expected_shape:

                            raise RuntimeError(
                                "NORM-PROXY INTERNAL SHAPE ERROR: "
                                f"num_selected={num_selected}, "
                                f"q_expand={tuple(q_expand.shape)}, "
                                f"k_selected={tuple(k_selected.shape)}, "
                                f"block_scores={tuple(block_scores.shape)}, "
                                f"expected={expected_shape}"
                            )

                        # =================================================
                        # CAUSAL MASK
                        # =================================================

                        q_positions = torch.arange(
                            qs,
                            qe,
                            device=q.device,
                        )

                        causal_invalid = (
                            k_indices[:, None, :]
                            > q_positions[
                                None,
                                :,
                                None,
                            ]
                        )

                        block_scores = (
                            block_scores.masked_fill(
                                causal_invalid,
                                float("-inf"),
                            )
                        )

                        # =================================================
                        # FLATTEN SELECTED BLOCKS
                        #
                        # [N,32,32]
                        #       ↓
                        # [32,N*32]
                        #
                        # This reshape is now mathematically guaranteed.
                        # =================================================

                        flat_scores = (
                            block_scores
                            .permute(
                                1,
                                0,
                                2,
                            )
                            .contiguous()
                            .reshape(
                                BLOCK_ROWS,
                                num_selected
                                * BLOCK_COLS,
                            )
                        )

                        # =================================================
                        # APPLY EXTERNAL ATTENTION MASK
                        # =================================================

                        if (
                            attention_mask
                            is not None
                        ):

                            if (
                                attention_mask.dim()
                                == 4
                            ):

                                mask_2d = (
                                    attention_mask[
                                        0,
                                        0,
                                        qs:qe,
                                        :key_len,
                                    ]
                                )

                                mask_selected = (
                                    torch.index_select(
                                        mask_2d,
                                        1,
                                        k_indices.reshape(
                                            -1
                                        ),
                                    )
                                )

                                flat_scores = (
                                    flat_scores
                                    + mask_selected
                                )

                            elif (
                                attention_mask.dim()
                                == 2
                            ):

                                valid_keys = (
                                    attention_mask[
                                        0,
                                        k_indices.reshape(
                                            -1
                                        ),
                                    ]
                                    == 0
                                )

                                flat_scores = (
                                    flat_scores.masked_fill(
                                        valid_keys[
                                            None,
                                            :,
                                        ],
                                        float("-inf"),
                                    )
                                )

                        # =================================================
                        # SAFETY AGAINST ALL -INF ROWS
                        # =================================================

                        finite_rows = (
                            torch.isfinite(
                                flat_scores
                            ).any(
                                dim=-1
                            )
                        )

                        if not bool(
                            finite_rows.all()
                        ):

                            bad_rows = torch.where(
                                ~finite_rows
                            )[0]

                            for local_r in (
                                bad_rows.tolist()
                            ):

                                global_r = (
                                    qs
                                    + local_r
                                )

                                legal = (
                                    k_indices
                                    <= global_r
                                )

                                positions = (
                                    legal.nonzero(
                                        as_tuple=False
                                    )
                                )

                                if positions.numel() > 0:

                                    block_id = int(
                                        positions[
                                            0,
                                            0,
                                        ].item()
                                    )

                                    key_local = int(
                                        positions[
                                            0,
                                            1,
                                        ].item()
                                    )

                                    flat_col = (
                                        block_id
                                        * BLOCK_COLS
                                        + key_local
                                    )

                                else:

                                    flat_col = 0

                                flat_scores[
                                    local_r,
                                    flat_col,
                                ] = 0.0

                        # =================================================
                        # SOFTMAX
                        # =================================================

                        weights = torch.softmax(
                            flat_scores.float(),
                            dim=-1,
                        ).to(
                            q.dtype
                        )

                        # =================================================
                        # GATHER V
                        # =================================================

                        v_selected = torch.index_select(
                            v_head,
                            0,
                            k_indices.reshape(-1),
                        )

                        v_selected = (
                            v_selected.reshape(
                                num_selected,
                                BLOCK_COLS,
                                head_dim,
                            )
                        )

                        # =================================================
                        # ATTENTION × V
                        # =================================================

                        v_flat = (
                            v_selected.reshape(
                                num_selected
                                * BLOCK_COLS,
                                head_dim,
                            )
                        )

                        block_output = torch.matmul(
                            weights,
                            v_flat,
                        )

                        # =================================================
                        # WRITE OUTPUT
                        # =================================================

                        output[
                            0,
                            h,
                            qs:qe,
                            :
                        ] = block_output

                # =================================================
                # MERGE HEADS
                # =================================================

                output = (
                    output
                    .transpose(
                        1,
                        2,
                    )
                    .contiguous()
                )

                output = output.reshape(
                    bsz,
                    q_len,
                    num_heads
                    * head_dim,
                )

                # =================================================
                # OUTPUT PROJECTION
                # =================================================

                output = self.o_proj(
                    output
                )

                # =================================================
                # SAME API AS LlamaAttention
                # =================================================

                return (
                    output,
                    None,
                )

            return proxy_forward

        # --------------------------------------------------------
        # Install instance-level forward.
        # --------------------------------------------------------

        attn.forward = types.MethodType(
            make_forward(layer_idx),
            attn,
        )

    # ============================================================
    # DONE
    # ============================================================

    print(
        "Norm-proxy VECTOR attention installed:",
        f"{BLOCK_ROWS}x{BLOCK_COLS}",
        f"target={CURRENT_PROXY_SPARSITY}%",
    )

    print(
        "QK strategy: batched selected-block matmul"
    )

    print(
        "Full QK matrix: NOT computed"
    )


# ============================================================
# INITIALIZE
# ============================================================

reset_proxy_stats()

print("=" * 70)
print("CELL 3 READY — CLEAN PATCH")
print("=" * 70)
print(
    "Method: ||Q_block||F × ||K_block||F → Top-K → exact selected QK"
)
print(
    f"Block size: {BLOCK_ROWS}x{BLOCK_COLS}"
)
print(
    "L1 QK ranking: DISABLED"
)
print(
    "Full QK matrix: DISABLED"
)
print(
    "Old instance-level attention patches: CLEARED ON INSTALL"
)
print("=" * 70)

CELL 3 READY — CLEAN PATCH
Method: ||Q_block||F × ||K_block||F → Top-K → exact selected QK
Block size: 32x32
L1 QK ranking: DISABLED
Full QK matrix: DISABLED
Old instance-level attention patches: CLEARED ON INSTALL


In [ ]:
# ============================================================
# CELL 4 — SMALL SANITY TEST — 64 TOKENS
# ============================================================
#
# Purpose:
#   Verify that the VECTORIZED NORM-PROXY attention path
#   is actually executed.
#
# IMPORTANT:
#   Block size = 32x32
#   Therefore the test MUST have >= 32 tokens.
#
# We force exactly 64 tokens so that:
#
#   Q length = 64
#   K length = 64
#
# giving a 2x2 block grid per attention head.
#
# ============================================================

TARGET_TEST_TOKENS = 64
TEST_SPARSITY = 90.0

print("=" * 70)
print("INSTALLING NORM-PROXY ATTENTION")
print("=" * 70)

install_norm_proxy_attention(
    model,
    TEST_SPARSITY,
)


# ============================================================
# CREATE TEST INPUT
# ============================================================

test_text = (
    "This is a standalone norm proxy attention test. "
    "Query and key block norms are used to select blocks "
    "before exact QK computation. "
    "The purpose of this test is to verify that the "
    "32 by 32 block based norm proxy attention path "
    "is actually executed correctly on the GPU."
)


# ------------------------------------------------------------
# Tokenize WITHOUT forcing a short sequence first
# ------------------------------------------------------------

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    add_special_tokens=True,
)

input_ids = inputs["input_ids"].to(device)


# ============================================================
# FORCE EXACTLY 64 TOKENS
# ============================================================

current_len = input_ids.shape[1]

if current_len < TARGET_TEST_TOKENS:

    # Repeat the token sequence until >= 64 tokens.
    repeat_count = (
        TARGET_TEST_TOKENS
        + current_len
        - 1
    ) // current_len

    input_ids = (
        input_ids[0]
        .repeat(repeat_count)
        [:TARGET_TEST_TOKENS]
        .unsqueeze(0)
        .to(device)
    )

else:

    input_ids = (
        input_ids[
            :,
            :TARGET_TEST_TOKENS
        ]
        .to(device)
    )


# ============================================================
# ATTENTION MASK
# ============================================================

attention_mask = torch.ones_like(
    input_ids
)


# ============================================================
# RESET STATISTICS
# ============================================================

reset_proxy_stats()


# ============================================================
# FORWARD PASS
# ============================================================

print()
print("=" * 70)
print("NORM PROXY SMALL TEST")
print("=" * 70)

print(
    "Input shape      :",
    tuple(input_ids.shape),
)

print(
    "Target sparsity  :",
    TEST_SPARSITY,
    "%",
)

print(
    "Block size       :",
    f"{BLOCK_ROWS}x{BLOCK_COLS}",
)

print()
print("Running one forward pass...")
print()


with torch.no_grad():

    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )


logits = out.logits


# ============================================================
# NUMERICAL CHECK
# ============================================================

finite = bool(
    torch.isfinite(logits).all()
)

nan_count = int(
    torch.isnan(logits).sum()
)

inf_count = int(
    torch.isinf(logits).sum()
)


# ============================================================
# PROXY STATISTICS
# ============================================================

candidate_blocks = int(
    proxy_stats["candidate_blocks"]
)

kept_blocks = int(
    proxy_stats["kept_blocks"]
)

threshold_count = int(
    proxy_stats["threshold_count"]
)


if candidate_blocks > 0:

    measured_sparsity = (
        1.0
        - kept_blocks / candidate_blocks
    ) * 100.0

else:

    measured_sparsity = None


# ============================================================
# NEXT TOKEN
# ============================================================

next_token_id = int(
    logits[
        0,
        -1,
        :
    ].argmax().item()
)

next_token = tokenizer.decode(
    [next_token_id]
)


# ============================================================
# RESULT
# ============================================================

print("=" * 70)
print("NORM PROXY SMALL TEST RESULT")
print("=" * 70)

print(
    "Forward pass       : PASS"
)

print(
    "Input shape        :",
    tuple(input_ids.shape),
)

print(
    "Logits shape       :",
    tuple(logits.shape),
)

print(
    "Logits finite      :",
    finite,
)

print(
    "NaN count          :",
    nan_count,
)

print(
    "Inf count          :",
    inf_count,
)

print(
    "Candidate blocks   :",
    candidate_blocks,
)

print(
    "Kept blocks        :",
    kept_blocks,
)

print(
    "Threshold count    :",
    threshold_count,
)


if measured_sparsity is not None:

    print(
        "Measured sparsity  :",
        f"{measured_sparsity:.4f}%",
    )

else:

    print(
        "Measured sparsity  : N/A"
    )


print(
    "Next token ID      :",
    next_token_id,
)

print(
    "Next token         :",
    repr(next_token),
)


# ============================================================
# HARD VALIDATION
# ============================================================

print()
print("=" * 70)
print("VALIDATION")
print("=" * 70)


if not finite:

    raise RuntimeError(
        "Small proxy forward produced NaN/Inf. "
        "STOP before LongBench."
    )


if candidate_blocks <= 0:

    raise RuntimeError(
        "Norm Proxy was NOT activated. "
        "Candidate blocks = 0."
    )


if kept_blocks <= 0:

    raise RuntimeError(
        "Norm Proxy selected zero blocks."
    )


if threshold_count <= 0:

    raise RuntimeError(
        "Norm Proxy threshold was not generated."
    )


print(
    "Input >= 32 tokens : PASS"
)

print(
    "Norm Proxy active  : PASS"
)

print(
    "Selected blocks    : PASS"
)

print(
    "Finite logits      : PASS"
)

print()
print("SMALL TEST: PASS")

INSTALLING NORM-PROXY ATTENTION
Restored original LlamaAttention.forward for 32 layers.
Norm-proxy VECTOR attention installed: 32x32 target=90.0%
QK strategy: batched selected-block matmul
Full QK matrix: NOT computed

NORM PROXY SMALL TEST
Input shape      : (1, 64)
Target sparsity  : 90.0 %
Block size       : 32x32

Running one forward pass...

NORM PROXY SMALL TEST RESULT
Forward pass       : PASS
Input shape        : (1, 64)
Logits shape       : (1, 64, 32000)
Logits finite      : True
NaN count          : 0
Inf count          : 0
Candidate blocks   : 3072
Kept blocks        : 1024
Threshold count    : 1024
Measured sparsity  : 66.6667%
Next token ID      : 297
Next token         : 'in'

VALIDATION
Input >= 32 tokens : PASS
Norm Proxy active  : PASS
Selected blocks    : PASS
Finite logits      : PASS

SMALL TEST: PASS


In [ ]:
# ============================================================
# CELL 5 — DENSE BASELINE + NORM-PROXY 50/70/80/90
# ============================================================
#
# EXPERIMENT STRUCTURE
#
#   Dense baseline
#       ↓
#   Norm Proxy 50%
#       ↓
#   Norm Proxy 70%
#       ↓
#   Norm Proxy 80%
#       ↓
#   Norm Proxy 90%
#
# IMPORTANT:
#   - Dense is run ONCE.
#   - The SAME 45 LongBench samples are used for every condition.
#   - No L1(QK) importance is calculated.
#   - No L1 ranking/comparison is performed.
#   - Norm Proxy uses only ||Q_block||F * ||K_block||F
#     to decide which QK blocks receive exact computation.
#
# ============================================================


# ============================================================
# PART A — COMMON LONGBENCH GENERATION
# ============================================================

def generate_longbench_answer(sample):
    task = sample["task"]

    prompt = dataset2prompt[task].format(**sample)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=SEQ_LEN,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    max_new_tokens = int(
        dataset2maxlen.get(
            task,
            MAX_NEW_TOKENS_DEFAULT,
        )
    )

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )


def run_longbench_eval(model_name):
    cmd = [
        sys.executable,
        "/content/LongBench/LongBench/eval.py",
        "--model",
        model_name,
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"LongBench evaluation failed for {model_name}"
        )

    return result.stdout


# ============================================================
# PART B — DENSE BASELINE
# ============================================================
#
# SMART RESUME LOGIC
#
# If a valid Dense baseline result already exists:
#
#     /content/pred/dense_baseline/result.json
#
# then:
#
#     -> DO NOT run Dense inference again
#     -> DO NOT regenerate predictions
#     -> simply load the previous result
#
# If result.json does not exist but Dense prediction .jsonl
# files already exist:
#
#     -> DO NOT rerun Dense inference
#     -> run LongBench evaluation only
#
# If neither exists:
#
#     -> run Dense inference normally
#
# ============================================================


DENSE_OUTPUT_NAME = "dense_baseline"

DENSE_DIR = os.path.join(
    "/content/pred",
    DENSE_OUTPUT_NAME,
)

DENSE_RESULT_PATH = os.path.join(
    DENSE_DIR,
    "result.json",
)


# ============================================================
# CHECK WHETHER DENSE BASELINE ALREADY EXISTS
# ============================================================

def load_existing_dense_result():

    if not os.path.exists(
        DENSE_RESULT_PATH
    ):
        return None

    try:

        with open(
            DENSE_RESULT_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            data = json.load(f)

        # ----------------------------------------------------
        # Make sure the result is actually usable.
        # ----------------------------------------------------

        if not isinstance(
            data,
            dict,
        ):

            return None

        if len(data) == 0:

            return None

        return data

    except Exception:

        return None


# ============================================================
# CHECK WHETHER DENSE PREDICTIONS ALREADY EXIST
# ============================================================

def dense_predictions_exist():

    if not os.path.isdir(
        DENSE_DIR
    ):

        return False

    jsonl_files = [
        f
        for f in os.listdir(
            DENSE_DIR
        )
        if f.endswith(".jsonl")
    ]

    return len(jsonl_files) > 0


# ============================================================
# RESTORE DENSE ATTENTION
# ============================================================

def restore_dense_attention():

    """
    Restore the original Hugging Face LlamaAttention.forward.
    """

    restore_original_attention(
        model
    )

    print(
        "Dense attention restored."
    )


# ============================================================
# DENSE PREDICTION GENERATION
# ============================================================

def export_dense_predictions():

    """
    Run exactly the same LongBench samples using dense attention.
    """

    restore_dense_attention()

    records = []

    for i, sample in enumerate(
        samples
    ):

        print(
            f"[{i + 1}/{len(samples)}] "
            f"DENSE {sample['task']}"
        )

        pred = generate_longbench_answer(
            sample
        )

        records.append({
            "pred": pred,
            "answers": sample["answers"],
            "task": sample["task"],
            "all_classes": sample.get(
                "all_classes",
                [],
            ),
            "length": sample.get(
                "length",
                None,
            ),
        })

    os.makedirs(
        DENSE_DIR,
        exist_ok=True,
    )

    grouped = defaultdict(list)

    for record in records:

        grouped[
            record["task"]
        ].append(record)

    for task, task_records in grouped.items():

        path = os.path.join(
            DENSE_DIR,
            f"{task}.jsonl",
        )

        with open(
            path,
            "w",
            encoding="utf-8",
        ) as f:

            for record in task_records:

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

        print(
            "Saved:",
            path,
        )

    return DENSE_DIR


# ============================================================
# SMART DENSE BASELINE
# ============================================================

print()
print("=" * 90)
print("DENSE BASELINE — SMART RESUME")
print("=" * 90)


# ------------------------------------------------------------
# CASE 1:
# Complete evaluated Dense result already exists.
# ------------------------------------------------------------

existing_dense_result = (
    load_existing_dense_result()
)


if existing_dense_result is not None:

    print()
    print(
        "Existing Dense baseline found."
    )

    print(
        "Path:",
        DENSE_RESULT_PATH,
    )

    print(
        "Dense inference: SKIPPED"
    )

    print(
        "Dense LongBench evaluation: SKIPPED"
    )

    dense_task_scores = (
        existing_dense_result
    )


# ------------------------------------------------------------
# CASE 2:
# Predictions exist, but result.json does not.
#
# Do NOT regenerate Dense predictions.
# Only evaluate them.
# ------------------------------------------------------------

elif dense_predictions_exist():

    print()
    print(
        "Existing Dense prediction files found."
    )

    print(
        "Dense inference: SKIPPED"
    )

    print(
        "Running LongBench evaluation only..."
    )

    run_longbench_eval(
        DENSE_OUTPUT_NAME
    )

    if not os.path.exists(
        DENSE_RESULT_PATH
    ):

        raise FileNotFoundError(
            "Dense evaluation finished, "
            "but result.json was not created: "
            f"{DENSE_RESULT_PATH}"
        )

    with open(
        DENSE_RESULT_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        dense_task_scores = json.load(
            f
        )


# ------------------------------------------------------------
# CASE 3:
# Nothing exists.
#
# Run Dense baseline normally.
# ------------------------------------------------------------

else:

    print()
    print(
        "No existing Dense baseline found."
    )

    print(
        "Running Dense inference..."
    )

    gc.collect()
    torch.cuda.empty_cache()

    export_dense_predictions()

    run_longbench_eval(
        DENSE_OUTPUT_NAME
    )

    if not os.path.exists(
        DENSE_RESULT_PATH
    ):

        raise FileNotFoundError(
            "Dense result not found: "
            f"{DENSE_RESULT_PATH}"
        )

    with open(
        DENSE_RESULT_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        dense_task_scores = json.load(
            f
        )


# ============================================================
# DENSE AVERAGE SCORE
# ============================================================

dense_average_score = (
    sum(
        dense_task_scores.values()
    )
    / len(
        dense_task_scores
    )
)


# ============================================================
# PRINT DENSE RESULT
# ============================================================

print()
print("=" * 70)
print("DENSE BASELINE RESULT")
print("=" * 70)

print(
    "Per-task:",
    dense_task_scores,
)

print(
    f"Average score: "
    f"{dense_average_score:.4f}"
)

print(
    "Dense baseline rerun: NO"
)
# ============================================================
# PART C — NORM PROXY PREDICTIONS
# ============================================================

def export_proxy_predictions(
    sparsity
):
    """
    Run the SAME LongBench samples using the norm-proxy attention.

    Proxy:
        ||Q_block||F * ||K_block||F

    Exact QK:
        calculated ONLY for blocks selected by the proxy.
    """

    install_norm_proxy_attention(
        model,
        sparsity,
    )

    output_name = (
        f"qk_norm_proxy_"
        f"{int(sparsity)}pct_32x32"
    )

    records = []

    for i, sample in enumerate(
        samples
    ):

        print(
            f"[{i + 1}/{len(samples)}] "
            f"PROXY-{sparsity}% "
            f"{sample['task']}"
        )

        pred = generate_longbench_answer(
            sample
        )

        records.append({
            "pred": pred,
            "answers": sample["answers"],
            "task": sample["task"],
            "all_classes": sample.get(
                "all_classes",
                [],
            ),
            "length": sample.get(
                "length",
                None,
            ),
        })

    save_dir = os.path.join(
        "/content/pred",
        output_name,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    grouped = defaultdict(list)

    for record in records:
        grouped[
            record["task"]
        ].append(record)

    for task, task_records in grouped.items():

        path = os.path.join(
            save_dir,
            f"{task}.jsonl",
        )

        with open(
            path,
            "w",
            encoding="utf-8",
        ) as f:

            for record in task_records:

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

        print(
            "Saved:",
            path,
        )

    candidate = proxy_stats[
        "candidate_blocks"
    ]

    kept = proxy_stats[
        "kept_blocks"
    ]

    if candidate > 0:
        measured_sparsity = (
            1.0
            - kept / candidate
        ) * 100.0
    else:
        measured_sparsity = float(
            "nan"
        )

    if (
        proxy_stats["threshold_count"]
        > 0
    ):
        average_threshold = (
            proxy_stats["threshold_sum"]
            / proxy_stats["threshold_count"]
        )
    else:
        average_threshold = float(
            "nan"
        )

    stats = {
        "target_sparsity": float(
            sparsity
        ),
        "measured_sparsity": float(
            measured_sparsity
        ),
        "candidate_blocks": int(
            candidate
        ),
        "kept_blocks": int(
            kept
        ),
        "average_threshold": float(
            average_threshold
        ),
        "attention_calls": int(
            proxy_stats[
                "attention_calls"
            ]
        ),
    }

    return (
        output_name,
        stats,
    )


# ============================================================
# PART D — 50 / 70 / 80 / 90 SWEEP
# ============================================================

proxy_results = {}

for sparsity in SPARSITY_LEVELS:

    print()
    print()
    print("#" * 90)
    print(
        f"NORM PROXY EXPERIMENT — "
        f"{sparsity}% SPARSITY"
    )
    print("#" * 90)

    gc.collect()
    torch.cuda.empty_cache()

    output_name, stats = (
        export_proxy_predictions(
            sparsity
        )
    )

    run_longbench_eval(
        output_name
    )

    result_path = os.path.join(
        "/content/pred",
        output_name,
        "result.json",
    )

    if not os.path.exists(
        result_path
    ):
        raise FileNotFoundError(
            f"Missing result: "
            f"{result_path}"
        )

    with open(
        result_path,
        "r",
        encoding="utf-8",
    ) as f:
        scores = json.load(f)

    average_score = (
        sum(scores.values())
        / len(scores)
    )

    stats["per_task"] = scores

    stats["average_score"] = float(
        average_score
    )

    stats["score_loss_vs_dense"] = float(
        dense_average_score
        - average_score
    )

    proxy_results[
        str(int(sparsity))
    ] = stats

    print()
    print("=" * 75)
    print(
        f"NORM PROXY RESULT — "
        f"{sparsity}%"
    )
    print("=" * 75)

    print(
        "Target sparsity   :",
        stats["target_sparsity"],
        "%",
    )

    print(
        "Measured sparsity :",
        stats["measured_sparsity"],
        "%",
    )

    print(
        "Candidate blocks  :",
        stats["candidate_blocks"],
    )

    print(
        "Kept blocks       :",
        stats["kept_blocks"],
    )

    print(
        "Average threshold :",
        stats["average_threshold"],
    )

    print(
        "Average score     :",
        stats["average_score"],
    )

    print(
        "Score loss vs dense:",
        stats[
            "score_loss_vs_dense"
        ],
    )

    print(
        "Per-task          :",
        stats["per_task"],
    )


# ============================================================
# PART E — FINAL SUMMARY
# ============================================================

print()
print("=" * 105)
print("DENSE vs NORM-PROXY — FINAL SUMMARY")
print("=" * 105)

print(
    f"{'Level':>10} | "
    f"{'Measured':>12} | "
    f"{'Avg Score':>12} | "
    f"{'Score Loss':>12} | "
    f"{'Candidate':>12} | "
    f"{'Kept':>12} | "
    f"{'Threshold':>14}"
)

print("-" * 105)

print(
    f"{'Dense':>10} | "
    f"{'-':>12} | "
    f"{dense_average_score:>12.4f} | "
    f"{0.0:>12.4f} | "
    f"{'-':>12} | "
    f"{'-':>12} | "
    f"{'-':>14}"
)

for sparsity in SPARSITY_LEVELS:

    r = proxy_results[
        str(int(sparsity))
    ]

    print(
        f"{int(sparsity):>9}% | "
        f"{r['measured_sparsity']:>11.4f}% | "
        f"{r['average_score']:>12.4f} | "
        f"{r['score_loss_vs_dense']:>12.4f} | "
        f"{r['candidate_blocks']:>12,} | "
        f"{r['kept_blocks']:>12,} | "
        f"{r['average_threshold']:>14.6f}"
    )


# ============================================================
# PART F — SAVE FINAL JSON
# ============================================================

final_output = {
    "experiment":
        "QK_norm_proxy_block_importance",
    "model": MODEL_NAME,
    "tasks": TASKS,
    "samples_per_task":
        SAMPLES_PER_TASK,
    "total_samples":
        len(samples),
    "block_size": [
        BLOCK_ROWS,
        BLOCK_COLS,
    ],
    "proxy": PROXY_NAME,
    "sparsity_levels":
        SPARSITY_LEVELS,

    "dense_baseline": {
        "average_score":
            float(
                dense_average_score
            ),
        "per_task":
            dense_task_scores,
        "output_name":
            DENSE_OUTPUT_NAME,
    },

    "results":
        proxy_results,
}

json_path = (
    "/content/"
    "qk_norm_proxy_sweep_dense_"
    "50_70_80_90.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_output,
        f,
        indent=2,
        ensure_ascii=False,
    )

print()
print("=" * 100)
print("EXPERIMENT COMPLETE")
print("=" * 100)

print(
    "Dense baseline rerun : YES"
)

print(
    "Norm Proxy levels    : "
    "50% / 70% / 80% / 90%"
)

print(
    "L1 branch            : NO"
)

print(
    "Saved:",
    json_path,
)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



DENSE BASELINE — SMART RESUME

Existing Dense baseline found.
Path: /content/pred/dense_baseline/result.json
Dense inference: SKIPPED
Dense LongBench evaluation: SKIPPED

DENSE BASELINE RESULT
Per-task: {'gov_report': 34.17, 'narrativeqa': 0.42, 'qasper': 1.08}
Average score: 11.8900
Dense baseline rerun: NO


##########################################################################################
NORM PROXY EXPERIMENT — 50.0% SPARSITY
##########################################################################################
Restored original LlamaAttention.forward for 32 layers.
Norm-proxy VECTOR attention installed: 32x32 target=50.0%
QK strategy: batched selected-block matmul
Full QK matrix: NOT computed
[1/45] PROXY-50.0% narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/45] PROXY-50.0% narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[43/45] PROXY-90.0% gov_report


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[44/45] PROXY-90.0% gov_report


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[45/45] PROXY-90.0% gov_report
Saved: /content/pred/qk_norm_proxy_90pct_32x32/narrativeqa.jsonl
Saved: /content/pred/qk_norm_proxy_90pct_32x32/qasper.jsonl
Saved: /content/pred/qk_norm_proxy_90pct_32x32/gov_report.jsonl
Evaluating on: ['gov_report.jsonl', 'narrativeqa.jsonl', 'qasper.jsonl']

/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


NORM PROXY RESULT — 90.0%
Target sparsity   : 90.0 %
Measured sparsity : 90.0 %
Candidate blocks  : 95846400
Kept blocks       : 9584640
Average threshold : 11898.396571180556
Average score     : 12.066666666666668
Score loss vs dense: -0.17666666666666764
Per-task          : {'gov_report': 34.67, 'narrativeqa': 0.43, 'qasper': 1.1}

DENSE vs NORM-PROXY — FINAL SUMMARY
     L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/QK_Norm_Proxy_Results"

!cp -r /content/pred \
      "/content/drive/MyDrive/QK_Norm_Proxy_Results/"

!cp -r /content/results \
      "/content/drive/MyDrive/QK_Norm_Proxy_Results/"

!cp /content/qk_norm_proxy_sweep_dense_50_70_80_90.json \
      "/content/drive/MyDrive/QK_Norm_Proxy_Results/"

In [ ]:
!ls -lh "/content/drive/MyDrive/QK_Norm_Proxy_Results"

total 11K
drwx------ 7 root root 4.0K Oct  6 16:51 pred
-rw------- 1 root root 2.3K Oct  6 16:51 qk_norm_proxy_sweep_dense_50_70_80_90.json
drwx------ 2 root root 4.0K Oct  6 16:51 results
